## Import Libraries

In [11]:
import gym
import torch
import torch.nn as nn
import torch.optim as optim
import random
import numpy as np
from collections import deque

## HyperParameters

In [12]:
EPISODES = 300
GAMMA = 0.99
LR = 0.001
EPSILON_START = 1.0
EPSILON_END = 0.01
EPSILON_DECAY = 0.995
BATCH_SIZE = 64
MEMORY_SIZE = 10000

## Environment 

In [13]:
env = gym.make("CartPole-v1")
state_size = env.observation_space.shape[0]
action_size = env.action_space.n

/usr/local/lib/python3.12/dist-packages/gym/core.py:317: DeprecationWarning: WARN: Initializing wrapper in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(
/usr/local/lib/python3.12/dist-packages/gym/wrappers/step_api_compatibility.py:39: DeprecationWarning: WARN: Initializing environment in old step API which returns one bool instead of two. It is recommended to set `new_step_api=True` to use new step API. This will be the default behaviour in future.
  deprecation(


## Q-Network 

In [14]:
class QNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc = nn.Sequential(
            nn.Linear(state_size, 64),
            nn.ReLU(),
            nn.Linear(64, 64),
            nn.ReLU(),
            nn.Linear(64, action_size)
        )

    def forward(self, x):
        return self.fc(x)

## Replay Memory

In [15]:
memory = deque(maxlen=MEMORY_SIZE)

## Initialize network & optimizer

In [16]:
q_net = QNetwork()
optimizer = optim.Adam(q_net.parameters(), lr=LR)
loss_fn = nn.MSELoss()

epsilon = EPSILON_START
reward_list = []

## Training Loop

In [17]:
for episode in range(EPISODES):
    # For Gym >=0.26 use: state, _ = env.reset()
    state = env.reset()  # returns just the state
    total_reward = 0
    done = False

    while not done:
        # Epsilon-greedy action
        if random.random() < epsilon:
            action = env.action_space.sample()
        else:
            state_tensor = torch.FloatTensor(state).unsqueeze(0)  # add batch dimension
            with torch.no_grad():
                action = torch.argmax(q_net(state_tensor)).item()

        # Step environment (Gym 0.26+)
        next_state, reward, done, info = env.step(action)

        # Optional: clip rewards
        reward = max(min(reward, 1.0), -1.0)

        # Store experience
        memory.append((state, action, reward, next_state, done))
        state = next_state
        total_reward += reward

        # Train only if enough memory
        if len(memory) >= BATCH_SIZE:
            batch = random.sample(memory, BATCH_SIZE)
            states_b, actions_b, rewards_b, next_states_b, dones_b = zip(*batch)

            states_b = torch.FloatTensor(states_b)
            actions_b = torch.LongTensor(actions_b).unsqueeze(1)
            rewards_b = torch.FloatTensor(rewards_b).unsqueeze(1)
            next_states_b = torch.FloatTensor(next_states_b)
            dones_b = torch.FloatTensor(dones_b).unsqueeze(1)

            # Compute target Q-values
            with torch.no_grad():
                q_next = torch.max(q_net(next_states_b), dim=1, keepdim=True)[0]
                q_target = rewards_b + GAMMA * q_next * (1 - dones_b)

            q_current = q_net(states_b).gather(1, actions_b)
            loss = loss_fn(q_current, q_target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

    # Decay epsilon
    epsilon = max(EPSILON_END, epsilon * EPSILON_DECAY)
    reward_list.append(total_reward)

    if episode % 20 == 0:
        avg_reward = np.mean(reward_list[-20:])
        print(f"Episode {episode}, Avg Reward: {avg_reward:.2f}, Epsilon: {epsilon:.2f}")

AttributeError: module 'numpy' has no attribute 'bool8'